# Find f0 from a harmonic snapshot

`harmonic_f0` finds the fundamental that best explains one frame of partials
as harmonics. This notebook starts from five synthetic peaks, looks at the
competing candidates, compares amplitude and equal weights, adds an outlier,
then runs the same search on cello peaks from `docs/f0.md`. No other tutorial
is needed first.

In [ ]:
import numpy as np
import pandas as pd
from plotly import graph_objects as go
import audiospylt as asp

## Five partials

Five peaks at 500.4, 565.0, 706.2, 918.2 and 988.8 Hz, with amplitudes 0.82,
0.55, 0.28, 0.20 and 0.35. For each candidate f0 between `fmin` and `fmax`,
`harmonic_f0` rounds every peak to its nearest harmonic and measures how far
it lies from it in cents; the result is the candidate with the lowest RMS of
those distances, each peak weighted by its amplitude.

In [ ]:
s = pd.DataFrame({"freq": [500.4, 565.0, 706.2, 918.2, 988.8],
                  "amp": [0.82, 0.55, 0.28, 0.20, 0.35]})
h = asp.harmonic_f0(s, fmin=35, fmax=150)
print(f"f0 {h.f0:.3f} Hz, RMS {h.rms_cents:.3f} cents")
print("harmonics:", h.harmonics)
print("cents from each harmonic:", h.cents.round(1))

The peaks are harmonics 7, 8, 10, 13 and 14 of 70.945 Hz. The strongest,
500.4 Hz, is the furthest off, 13.1 cents sharp.

## Candidates

`candidate_f0` and `candidate_rms` hold every local best the search found.
Before you run the cell: which other candidate scores as well as 70.945 Hz?

In [ ]:
fig = go.Figure(go.Scatter(x=h.candidate_f0, y=h.candidate_rms, mode="markers"))
fig.update_xaxes(type="log", title="candidate f0 (Hz)")
fig.update_yaxes(title="RMS (cents)")
asp.nb.show(fig)
candidates = pd.DataFrame({"f0": h.candidate_f0, "rms_cents": h.candidate_rms})
candidates.nsmallest(3, "rms_cents").round(3)

35.473 Hz, an octave lower, ties at 10.132 cents: every harmonic of 70.945 Hz
is also a harmonic of half that frequency (7 becomes 14, and so on). Of all
candidates within `tol_cents` (3 cents by default) of the lowest RMS,
`harmonic_f0` returns the highest.

## Amplitude or equal weights

`weight="equal"` counts every peak the same.

In [ ]:
for weight in ("amp", "equal"):
    hw = asp.harmonic_f0(s, fmin=35, fmax=150, weight=weight)
    print(f"{weight}: f0 {hw.f0:.3f} Hz, RMS {hw.rms_cents:.3f} cents, cents {hw.cents.round(1)}")

Equal weights give 70.797 Hz. With amplitude weights the strongest peak counts
most, so f0 moves up to 70.945 Hz: that peak comes closer to its harmonic
(13.1 instead of 16.8 cents) and the four weaker ones move further off (about
−8 instead of −4 cents).

## An outlier

Add a sixth peak at 1234.0 Hz, once at amplitude 0.2 and once at 0.05. Before
you run the cell: which weight keeps f0 near 71 Hz?

In [ ]:
for amp in (0.2, 0.05):
    so = pd.concat([s, pd.DataFrame({"freq": [1234.0], "amp": [amp]})], ignore_index=True)
    for weight in ("amp", "equal"):
        ho = asp.harmonic_f0(so, fmin=35, fmax=150, weight=weight)
        print(f"amp {amp}, {weight}: f0 {ho.f0:.3f} Hz, RMS {ho.rms_cents:.3f} cents, "
              f"outlier on harmonic {ho.harmonics[-1]} at {ho.cents[-1]:+.1f} cents")

At amplitude 0.2 both weights drop an octave, to 35.455 Hz and 35.375 Hz,
where 1234.0 Hz fits harmonic 35. At 0.05, amplitude weights keep 70.981 Hz:
the outlier lies 38.8 cents off harmonic 17, and the RMS rises only from
10.132 to 11.598 cents. Equal weights still drop to 35.375 Hz.

## Cello peaks and `tol_cents`

The same call as in `docs/f0.md`: the cello plays D2; take 1.5–2.5 s, keep the
peaks that stand 10 dB above their surroundings and lie within 40 dB of the
strongest, and search 35–150 Hz. Before you run the cell: which candidate has
the lowest RMS?

In [ ]:
x, sr = asp.load_audio("../samples/cello_arco_d_sul_c_01.wav", mono=True)
spec = asp.spectrum(asp.trim(x, sr, 1.5, 2.5), sr)
peaks = asp.find_peaks(spec, min_db=None, prominence_db=10, rel_db=-40)
print(f"{len(peaks)} peaks")
for tol in (0, 3.0):
    hc = asp.harmonic_f0(peaks, fmin=35, fmax=150, tol_cents=tol)
    print(f"tol_cents={tol}: f0 {hc.f0:.2f} Hz, RMS {hc.rms_cents:.2f} cents")

37.20 Hz, an octave below the played note, has the lowest RMS (17.23 cents),
and `tol_cents=0` returns it. 74.40 Hz, the D2, scores 17.79 cents, within
the default 3 cents of the lowest, and is the higher of the two, so the
default returns it.